# Pipeline check: one clear Sentinel-2 chip over Thessaloniki port

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anefelo/anefelo-notebooks/blob/main/notebooks/0000-pipeline-check/pipeline_check.ipynb)

**Question.** Can this repository's runner reach open satellite data, compute on it and return figures and numbers? This notebook is plumbing, not a post: it finds the clearest recent Sentinel-2 scene over the port, reads a 6 × 5.5 km window straight from the cloud-optimised files and maps NDVI.

**Runs in** about 1–2 minutes on a free Colab CPU. No accounts or keys needed.

**Data:** Sentinel-2 Level-2A surface reflectance via the Earth Search STAC API (Element 84). Provenance in section 2.

## 0. Setup

In [1]:
%pip install -q "pystac-client>=0.8" "rasterio>=1.4" "matplotlib>=3.8"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime as dt
import json
import os
import platform
import random
import sys
from importlib import metadata
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR, RES_DIR = Path("figures"), Path("results")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})


def get_secret(name):
    """Colab: the Secrets panel (key icon). GitHub Actions: a repository secret of the same name."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name) or None


SUMMARY = {}


def record(key, value, unit=None, note=None):
    """Every number the post cites is recorded here and written to results/summary.json."""
    if isinstance(value, (np.floating, np.integer)):
        value = value.item()
    SUMMARY[key] = {"value": value, "unit": unit, "note": note}
    return value


def save_figure(fig, name, credit):
    """Saves figures/<name>.png with the data credit printed on the image."""
    fig.text(0.01, -0.01, credit, fontsize=8, color="0.35", ha="left", va="top")
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("saved", path)
    return path

## 1. Parameters

In [3]:
NOTEBOOK_NAME = "pipeline_check.ipynb"
STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-c1-l2a"
# Port of Thessaloniki: (min_lon, min_lat, max_lon, max_lat), WGS84
BBOX = (22.88, 40.61, 22.95, 40.66)
END = dt.date.today()
START = END - dt.timedelta(days=90)
MAX_CLOUD = 20  # percent, scene level
INPUTS = {"stac_url": STAC_URL, "collection": COLLECTION, "bbox": BBOX,
          "start": str(START), "end": str(END), "max_cloud_pct": MAX_CLOUD}

## 2. Data and provenance

| Dataset | Provider and access | Collection / version | Licence and credit | Citation |
|---|---|---|---|---|
| Sentinel-2 Level-2A surface reflectance (10 m red, green, blue, NIR; 20 m scene classification) | Copernicus / ESA, indexed by Element 84 Earth Search, https://earth-search.aws.element84.com/v1 | `sentinel-2-c1-l2a` | Free and open Copernicus data; credit "Contains modified Copernicus Sentinel data [year]" | — |

Accessed on the date printed in the last cell.

## 3. Method

1. Search the catalogue for scenes over the port in the last 90 days with under 20% cloud.
2. Take the clearest one and read only the pixels inside the box (no full-scene download).
3. Convert raw values to surface reflectance using the scale and offset the catalogue declares.
4. Drop cloud, shadow and nodata pixels using the scene classification layer.
5. Compute NDVI = (NIR − red) / (NIR + red) and summarise it.

In [4]:
import pystac_client
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

catalog = pystac_client.Client.open(STAC_URL)
search = catalog.search(
    collections=[COLLECTION],
    bbox=BBOX,
    datetime=f"{START}/{END}",
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    max_items=100,
)
items = list(search.items())
assert items, f"No {COLLECTION} scenes under {MAX_CLOUD}% cloud between {START} and {END}"
item = min(items, key=lambda it: it.properties["eo:cloud_cover"])
print(f"{len(items)} candidate scenes; using {item.id} "
      f"({item.properties['datetime'][:10]}, {item.properties['eo:cloud_cover']:.1f}% cloud)")


def scale_offset(asset):
    """Reflectance = DN * scale + offset, as declared by the STAC raster extension."""
    bands = asset.extra_fields.get("raster:bands") or [{}]
    return bands[0].get("scale", 1.0), bands[0].get("offset", 0.0)


def read_window(href, bbox, out_shape=None, resampling=Resampling.nearest):
    """Reads only the pixels inside bbox from a cloud-optimised GeoTIFF."""
    with rasterio.open(href) as src:
        bounds = transform_bounds("EPSG:4326", src.crs, *bbox, densify_pts=21)
        window = from_bounds(*bounds, transform=src.transform)
        return src.read(1, window=window, out_shape=out_shape, resampling=resampling).astype("float32")


def reflectance(item, key, bbox, out_shape=None):
    asset = item.assets[key]
    dn = read_window(asset.href, bbox, out_shape=out_shape)
    scale, offset = scale_offset(asset)
    out = dn * scale + offset
    out[dn == 0] = np.nan  # 0 is nodata in Sentinel-2 L2A
    return out

/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail']
  warnings.warn(


44 candidate scenes; using S2B_T34TFL_20260914T092025_L2A (2026-09-14, 0.0% cloud)


In [5]:
red = reflectance(item, "red", BBOX)
shape = red.shape
green = reflectance(item, "green", BBOX, out_shape=shape)
blue = reflectance(item, "blue", BBOX, out_shape=shape)
nir = reflectance(item, "nir", BBOX, out_shape=shape)
scl = read_window(item.assets["scl"].href, BBOX, out_shape=shape)  # 20 m, resampled to the 10 m grid

# Scene classification: 4 vegetation, 5 bare soil, 6 water, 7 unclassified; the rest is cloud, shadow, snow or nodata
valid = np.isin(scl, [4, 5, 6, 7]) & np.isfinite(red) & np.isfinite(nir)
ndvi = np.where(valid, (nir - red) / (nir + red + 1e-6), np.nan)

record("scene_id", item.id)
record("scene_date", item.properties["datetime"][:10])
record("scene_cloud_cover", round(float(item.properties["eo:cloud_cover"]), 2), "%", "scene-level, from STAC metadata")
record("chip_pixels", int(valid.size), "pixels", "10 m pixels inside the bounding box")
record("valid_fraction", round(float(valid.mean()), 4), "share", "clear land or water pixels per the SCL band")
record("mean_ndvi_valid", round(float(np.nanmean(ndvi)), 4), "NDVI", "mean over valid pixels")
record("vegetated_share", round(float(np.nanmean(ndvi[valid] > 0.4)), 4), "share", "valid pixels with NDVI > 0.4")

0.0623

## 4. Results

In [6]:
def stretch(x, lo=0.0, hi=0.3):
    return np.clip((x - lo) / (hi - lo), 0, 1)

rgb = np.dstack([stretch(red), stretch(green), stretch(blue)])
rgb = np.where(np.isfinite(rgb), rgb, 1.0)
year = item.properties["datetime"][:4]

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
axes[0].imshow(rgb)
axes[0].set_title(f"True colour, {item.properties['datetime'][:10]}")
im = axes[1].imshow(ndvi, cmap="YlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title(f"NDVI (mean {SUMMARY['mean_ndvi_valid']['value']:.2f} over clear pixels)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes[1], shrink=0.8, label="NDVI")
fig.suptitle("Pipeline check: Thessaloniki port, Sentinel-2 L2A at 10 m", fontsize=12)
save_figure(fig, "fig1_thessaloniki_truecolor_ndvi", credit=f"Contains modified Copernicus Sentinel data {year}")
plt.show()

saved figures/fig1_thessaloniki_truecolor_ndvi.png


In [7]:
summary = {
    "meta": {
        "notebook": NOTEBOOK_NAME,
        "generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "inputs": INPUTS,
    },
    "values": SUMMARY,
}
(RES_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))

{
  "meta": {
    "notebook": "pipeline_check.ipynb",
    "generated_utc": "2026-10-11T11:45:27+00:00",
    "inputs": {
      "stac_url": "https://earth-search.aws.element84.com/v1",
      "collection": "sentinel-2-c1-l2a",
      "bbox": [
        22.88,
        40.61,
        22.95,
        40.66
      ],
      "start": "2026-07-13",
      "end": "2026-10-11",
      "max_cloud_pct": 20
    }
  },
  "values": {
    "scene_id": {
      "value": "S2B_T34TFL_20260914T092025_L2A",
      "unit": null,
      "note": null
    },
    "scene_date": {
      "value": "2026-09-14",
      "unit": null,
      "note": null
    },
    "scene_cloud_cover": {
      "value": 0.0,
      "unit": "%",
      "note": "scene-level, from STAC metadata"
    },
    "chip_pixels": {
      "value": 343072,
      "unit": "pixels",
      "note": "10 m pixels inside the bounding box"
    },
    "valid_fraction": {
      "value": 0.9973,
      "unit": "share",
      "note": "clear land or water pixels per the SCL band"

## 5. What this does not show

- One date only: no change over time, so nothing about port activity.
- 10 m pixels: ships, cranes and containers are mixed pixels at this resolution.
- The scene classification layer misses some thin cloud and haze; NDVI over water is not meaningful.

In [8]:
# Environment, so a reader can match versions if a number differs
packages = ["pystac-client", "pystac", "rasterio", "numpy", "matplotlib"]
print("generated (UTC):", dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))
print("python:", platform.python_version(), "| colab:", IN_COLAB)
for name in packages:
    try:
        print(f"{name}: {metadata.version(name)}")
    except metadata.PackageNotFoundError:
        print(f"{name}: not installed")

generated (UTC): 2026-10-11T11:45:27+00:00
python: 3.11.17 | colab: False
pystac-client: 0.9.0
pystac: 1.15.2
rasterio: 1.4.4
numpy: 2.4.6
matplotlib: 3.11.2
